In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import optax
from flax import nnx

from sklearn.datasets import make_regression
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split

from deepordinal.jax import OrdinalOutput, ordinal_loss

In [ ]:
x, y = make_regression(n_samples=10000, n_features=8, n_informative=6, random_state=0)

qt = QuantileTransformer()
y = qt.fit_transform(y.reshape(-1, 1))[:, 0]
y = np.floor(y * 4).astype(np.int32)  # 4 ordinal classes: 0, 1, 2, 3
y = np.clip(y, 0, 3)

x_train, x_val, y_train, y_val = train_test_split(
    x.astype(np.float32), y, test_size=0.25, random_state=0
)
print("Class distribution:", np.bincount(y_train))

In [ ]:
class Model(nnx.Module):
    def __init__(self, rngs):
        self.hidden = nnx.Linear(8, 16, rngs=rngs)
        self.ordinal = OrdinalOutput(input_dim=16, output_dim=4, rngs=rngs)

    def features(self, x):
        return jax.nn.relu(self.hidden(x))

    def __call__(self, x):
        return self.ordinal(self.features(x))


model = Model(nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)
n_params = sum(p.size for p in jax.tree_util.tree_leaves(nnx.state(model, nnx.Param)))
print(f"Trainable parameters: {n_params}")

In [ ]:
def loss_fn(model, x, y):
    logits = model.ordinal.logit(model.features(x))
    thresholds = model.ordinal.interior_thresholds  # ordered by construction
    return ordinal_loss(logits, y, thresholds)


@nnx.jit
def train_step(model, optimizer, x, y):
    loss, grads = nnx.value_and_grad(loss_fn)(model, x, y)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    return loss_fn(model, x, y)


def batches(x, y, batch_size, rng=None):
    idx = np.arange(len(x)) if rng is None else rng.permutation(len(x))
    for start in range(0, len(x), batch_size):
        b = idx[start:start + batch_size]
        yield x[b], y[b]


rng = np.random.default_rng(0)
epochs = 50
train_losses, val_losses = [], []

for epoch in range(epochs):
    epoch_loss = [train_step(model, optimizer, xb, yb) for xb, yb in batches(x_train, y_train, 16, rng)]
    train_losses.append(float(np.mean(epoch_loss)))

    v_loss = [eval_step(model, xb, yb) for xb, yb in batches(x_val, y_val, 16)]
    val_losses.append(float(np.mean(v_loss)))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
probs = model(jnp.asarray(x_val))
preds = np.asarray(probs.argmax(axis=1))
accuracy = np.mean(preds == y_val)
print(f"Validation accuracy: {accuracy:.4f}")

In [ ]:
plt.plot(train_losses, '--', label='train_loss')
plt.plot(val_losses, label='val_loss')
plt.title('Training and validation loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend()
plt.show()